# C.7 Reward 怎麼改變輸出機率？

# C：推理步驟、驗證與測試時算力

前置：下一字生成、SFT與標準答案驗證。像解數學題：寫出步驟、多試幾種辦法、檢查答案，是三個不同動作。真實推理質量需要trained模型；默認例子用可驗證的數字，避免把長文字當思考保證。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：多試幾次，還要知道怎麼選**

候選包含正解，不代表最後答案已經選中。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/reasoning.svg")))

**先想一想：**reward正於baseline，梯度下降應增加還是減少該動作概率？

把得到獎勵的採樣動作log概率往上推，是最小policy gradient直覺。單次很吵，加入baseline可降低波動；真正RLVR需很多樣本與獨立驗證。

**把直覺寫成數學：**L=−(reward−baseline)·logp(sampled_action)，reward不反向傳播。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
logits = torch.tensor([0.0, 0.0], requires_grad=True)
action = 1
reward = 1.0
baseline = 0.5
loss = -(reward - baseline) * logits.log_softmax(0)[action]
loss.backward()
print("梯度", logits.grad)
with torch.no_grad():
    updated = logits - 0.1 * logits.grad
print("前後概率", logits.detach().softmax(0), updated.softmax(0))
assert updated.softmax(0)[1] > 0.5

**如何讀結果：**這是兩動作一次手算更新，不宣稱訓練完成；held-out成功率、長度與reward漏洞需另測。

**只改一件事：**只讓reward低於baseline，預先判斷方向。
